# RAG Query Engine and Persistent Chatbot

This notebook is now organized like a small RAG application instead of a pile of helper functions. The same behavior is kept, but the responsibilities are grouped into three modules:

1. `ChromaKnowledgeBase`: reconnects to ChromaDB, rebuilds the LlamaIndex vector index, rebuilds BM25, and creates the hybrid retriever.
2. `JsonChatSessionStore`: saves and loads chat memory as one JSON file per chat in the `session` folder.
3. `RagNewsChatbot`: exposes the product-facing actions: ask one question, open a chat, continue a chat, inspect history, and show sources.

```mermaid
flowchart LR
    U[User question] --> APP[RagNewsChatbot]
    APP --> HR[HybridRetriever]
    HR --> VS[Semantic search\nChroma vector index]
    HR --> BM[Keyword search\nBM25]
    VS --> RRF[Reciprocal rank fusion]
    BM --> RRF
    RRF --> LLM[Gemma 3 1B via Ollama]
    LLM --> A[Answer + sources]
    APP <--> MEM[ChatMemoryBuffer]
    MEM <--> JSON[session/*.json]
```

The important product idea is separation of concerns: retrieval, generation, and persistence are related, but they should not be tangled together.

In [ ]:
# Run this once if the LlamaIndex integrations are missing, then restart the kernel.
%pip install llama-index-vector-stores-chroma llama-index-retrievers-bm25 llama-index-llms-ollama

In [15]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "app.py").exists() and (project_root.parent / "app.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
import src.rag.chatbot as rag_chatbot_module
import src.rag.factory as rag_factory_module
import src.rag.session_store as rag_session_store_module

importlib.reload(rag_session_store_module)
importlib.reload(rag_chatbot_module)
importlib.reload(rag_factory_module)

from src.rag import (
    DEFAULT_COLLECTION_NAME,
    DEFAULT_EMBED_MODEL_NAME,
    DEFAULT_OLLAMA_MODEL,
    ChromaKnowledgeBase,
    JsonChatSessionStore,
    create_rag_app,
    default_paths,
    print_sources,
)

In [16]:
PROJECT_ROOT = project_root
paths = default_paths(PROJECT_ROOT)
NEWS_DIR = paths.news_dir
CHROMA_DIR = paths.chroma_dir
SESSION_DIR = paths.session_dir
COLLECTION_NAME = DEFAULT_COLLECTION_NAME
EMBED_MODEL_NAME = DEFAULT_EMBED_MODEL_NAME
OLLAMA_MODEL = DEFAULT_OLLAMA_MODEL

print(f"Project root: {PROJECT_ROOT}")
print(f"News source folder: {NEWS_DIR}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Session folder: {SESSION_DIR}")
print(f"Collection name: {COLLECTION_NAME}")
print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"LLM model: {OLLAMA_MODEL}")

Project root: C:\Program Files\Studying\coding\RAG_project
News source folder: C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Session folder: C:\Program Files\Studying\coding\RAG_project\session
Collection name: news_chat
Embedding model: BAAI/bge-small-en-v1.5
LLM model: gemma3:1b


In [ ]:
# Retrieval, fusion, and knowledge-base classes are now centralized in src/rag/.
# This notebook imports and uses the same implementation as app.py.

In [ ]:
# Session persistence and chatbot facade now come from src/rag/.
# This removes duplicate maintenance between the notebook and app.py.

## Build the Application Objects

Run the next cell once per notebook session. It creates:

- `knowledge_base`: reconnects to the persisted `news_chat` ChromaDB collection
- `session_store`: reads and writes chat JSON files under `session/`
- `rag_app`: the single object you use for query, chat, sources, and history

This is the main readability improvement: examples below call methods on `rag_app` instead of manually coordinating many functions and global variables.

## Run the RAG App

The next cells demonstrate the same functionality as before, but through one object: `rag_app`.

Use `rag_app.ask(...)` for one-off questions. Use `rag_app.open_chat(...)` and `rag_app.chat(...)` for multi-turn conversations with persistent memory.

### Initialize the App

In [17]:
# Run this once after reopening the notebook.
# It reconnects to ChromaDB, rebuilds semantic + BM25 retrieval, and prepares chat persistence.
rag_app = create_rag_app(
    chroma_dir=CHROMA_DIR,
    session_dir=SESSION_DIR,
    news_dir=NEWS_DIR,
    collection_name=COLLECTION_NAME,
    embed_model_name=EMBED_MODEL_NAME,
    ollama_model=OLLAMA_MODEL,
    final_top_k=5,
)
knowledge_base = rag_app.knowledge_base

print(f"Stored vector count: {knowledge_base.count()}")
print("RAG app is ready.")

Stored vector count: 1499
RAG app is ready.


### Single-turn RAG: use this when the question does not need chat history.

In [ ]:
sample_query = "Who is Sanae Takaichi, and what is her relationship with Donald Trump?"
query_response = rag_app.ask(sample_query)

print("-" * 70)
print_sources(query_response)

### Multi-turn RAG With Saved Chat History

`rag_app.open_chat(...)` loads the saved JSON file when it exists, or creates a new one when it does not. Every `rag_app.chat(...)` call saves the updated conversation automatically.

In [ ]:
# Create or load one persistent chat session.
# If session/Sanae_Takaichi.json exists, this restores its previous messages.
chat_id = rag_app.open_chat("Sanae Takaichi")

chat_response = rag_app.chat(
    chat_id=chat_id,
    message="Who is Sanae Takaichi?",
)
print_sources(chat_response)

In [ ]:
# Follow-up question: this uses the same saved chat memory, so "she" refers to Sanae Takaichi.
follow_up_response = rag_app.chat(
    chat_id=chat_id,
    message="How long did she talk with Donald Trump during a phone call?",
)
print_sources(follow_up_response)

In [ ]:
# Inspect the current in-memory history for this chat.
rag_app.show_history(chat_id)

In [ ]:
# List saved chat files on disk.
rag_app.list_saved_chats()

### Session Management Helpers

Use these APIs to inspect and manage persisted chat sessions:
- `rag_app.count_chat_ids()`
- `rag_app.list_chat_ids()`
- `rag_app.rename_chat(old_chat_id, new_chat_id)`
- `rag_app.delete_chat(chat_id)`

In [22]:
# Count and list chat ids currently saved on disk.
print("Saved chat count:", rag_app.count_chat_ids())
print("Saved chat ids:", rag_app.list_chat_ids())

Saved chat count: 1
Saved chat ids: ['Sanae Takaichi']


In [21]:
for delete_chat_id in ['cb1dd22b-0a72-4c57-b1c3-a0c89ff35339', '4bd0ee41-0fb8-4e99-af51-697941345131', '6723cfa1-69f8-476f-a5e3-5a3b68f3195c', 'e57cee56-0b5b-48c8-929c-cf2c4c224fe6', '75976806-8c81-466b-a15a-e16e3fbf753a', 'chat_20260623_013811', '43b8541d-e329-4591-8316-534efa6a67be']:
    rag_app.delete_chat(delete_chat_id, close_open_session=True, missing_ok=False)

### Reopen an Existing Chat

After reopening the notebook, run the import/config/module cells and the app initialization cell first. Then `rag_app.open_chat("Sanae Takaichi")` restores the saved conversation from `session/Sanae_Takaichi.json`.

In [4]:
loaded_chat_id = rag_app.open_chat("Sanae Takaichi")

In [5]:
rag_app.show_history(loaded_chat_id)

user: Who is Sanae Takaichi?
assistant: According to the provided context, Sanae Takaichi is the Prime Minister of Japan who won a landslide election in snap elections on Sunday. She is viewed as a conservative and security hawk.

Here’s a breakdown of what the context tells us:

*   **Prime Minister:** He is the leader of Japan.
*   **Recent Political Situation:** He was recently criticized by China, leading to a sharp diplomatic backlash.
*   **Policy:** He’s known for accelerating Japan’s defense strategy, including increased military spending and security cooperation with allies.


----------------------------------------
user: How long did she talk with Donald Trump during a phone call?
assistant: The provided context states that Trump and Takaichi “exchanged views mainly on the Indo-Pacific region and confirmed the close cooperation between Japan and the United States” during their phone call.

It doesn’t specify the duration of the call.
----------------------------------------


In [ ]:
# Continue a loaded chat without rerunning the original conversation cells.
reloaded_response = rag_app.chat(
    chat_id=loaded_chat_id,
    message="What is the latest news about Sanae Takaichi ?",
)
print_sources(reloaded_response)